# Empirical Analysis: META Stock Vest Trajectory & Optimal Sell Timing

**Author**: Antigravity Quantitative Research  
**Target Ticker**: `META` (Meta Platforms, Inc.)  
**Benchmark**: `SPY` (S&P 500 ETF)  
**Trading Window Constraint**: **Window Closes at $V+10$** (10 Trading Days Post-Vest)  
**Timeframes Analyzed**:
1. **Since 2014 (All-Time - 50 Vests)**
2. **Last 10 Years (2016–2026 - 42 Vests)**
3. **Last 5 Years (2021–2026 - 22 Vests)**

---

## Executive Summary & Key Strategic Directives

Meta Platforms employees receive quarterly equity vestings (Feb 15, May 15, Aug 15, Nov 15). Employees can execute open-market sales starting on **$V+1$** (First Sellable Day).

> [!IMPORTANT]
> **Trading Window Boundary**: The internal employee trading window typically **closes at $V+10$** (2 weeks post-vest). Therefore, execution strategies MUST be optimized within the active window $t \in [1, 10]$.

### Core Research Questions:
1. Is it better to sell at **Market Open ($O_1$)** or **Market Close ($C_1$)** on $V+1$?
2. What is the price trajectory from $V+0$ to the window cutoff at $V+10$?
3. How do results differ across historical timeframes (2014+, Last 10Y, Last 5Y)?


In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")


Libraries imported successfully.


## 1. Multi-Timeframe Data Pipeline & Vest Calendar Engine

In [2]:
# Load calculated multi-timeframe results
import os
json_filename = 'vest_analysis_results.json'
if not os.path.exists(json_filename) and os.path.exists(os.path.join('Projects/meta-vest-analysis/repo', json_filename)):
    json_filename = os.path.join('Projects/meta-vest-analysis/repo', json_filename)

with open(json_filename, 'r') as f:
    results_data = json.load(f)

print("Available Timeframes:", list(results_data.keys()))
print(f"All Time Vests: {results_data['all']['n_events']}")
print(f"Last 10Y Vests: {results_data['last10y']['n_events']}")
print(f"Last 5Y Vests:  {results_data['last5y']['n_events']}")


Available Timeframes: ['all', 'last10y', 'last5y', 'current_vest_cycle']
All Time Vests: 50
Last 10Y Vests: 42
Last 5Y Vests:  22


## 2. Separate OPEN and CLOSE Price Trajectories (with 95% Confidence Intervals & V+10 Cutoff Line)

Below we plot separate charts for **Market Open ($O_t$)** and **Market Close ($C_t$)** relative to Vest Day $V+0$ Close ($C_0$).
The vertical dashed line marks **$V+10$** where the employee trading window closes.


In [3]:
def plot_open_close_separate(tf_key='all', tf_title='Since 2014 (All-Time)'):
    days_data = results_data[tf_key]['summary_days']
    days_x = [d['day_label'] for d in days_data]
    
    # 1. OPEN Chart
    o_mean = [d['open_mean'] for d in days_data]
    o_med = [d['open_median'] for d in days_data]
    o_lower = [d['open_ci_lower'] for d in days_data]
    o_upper = [d['open_ci_upper'] for d in days_data]
    
    fig_o = go.Figure()
    fig_o.add_trace(go.Scatter(
        x=days_x + days_x[::-1],
        y=o_upper + o_lower[::-1],
        fill='toself',
        fillcolor='rgba(255, 159, 67, 0.15)',
        line=dict(color='rgba(0,0,0,0)'),
        name='95% Confidence Interval (SE)',
        showlegend=True
    ))
    fig_o.add_trace(go.Scatter(x=days_x, y=o_mean, mode='lines+markers', name='OPEN Mean Return (%)',
                               line=dict(color='#ff9f43', width=3), marker=dict(size=7)))
    fig_o.add_trace(go.Scatter(x=days_x, y=o_med, mode='lines+markers', name='OPEN Median Return (%)',
                               line=dict(color='#f59e0b', width=2.5, dash='dash'), marker=dict(size=6)))
    
    fig_o.add_shape(type="line", x0="V+10", x1="V+10", y0=0, y1=1, yref="paper",
                    line=dict(color="#ef4444", width=3, dash="dash"))
    fig_o.add_annotation(x="V+10", y=1.02, yref="paper", text="Trading Window Closes (V+10)",
                         showarrow=False, font=dict(color="#ef4444", size=12, family="Inter"))
    fig_o.add_hline(y=0, line_dash="dash", line_color="white")
    
    fig_o.update_layout(
        title=f"MARKET OPEN Price Trajectory & 95% CI ({tf_title})",
        xaxis_title="Trading Days Post-Vest",
        yaxis_title="Return Relative to V+0 Close (%)",
        template="plotly_dark",
        height=480
    )
    # 2. CLOSE Chart
    c_mean = [d['close_mean'] for d in days_data]
    c_med = [d['close_median'] for d in days_data]
    c_lower = [d['close_ci_lower'] for d in days_data]
    c_upper = [d['close_ci_upper'] for d in days_data]
    
    fig_c = go.Figure()
    fig_c.add_trace(go.Scatter(
        x=days_x + days_x[::-1],
        y=c_upper + c_lower[::-1],
        fill='toself',
        fillcolor='rgba(0, 242, 254, 0.15)',
        line=dict(color='rgba(0,0,0,0)'),
        name='95% Confidence Interval (SE)',
        showlegend=True
    ))
    fig_c.add_trace(go.Scatter(x=days_x, y=c_mean, mode='lines+markers', name='CLOSE Mean Return (%)',
                               line=dict(color='#00f2fe', width=3), marker=dict(size=7)))
    fig_c.add_trace(go.Scatter(x=days_x, y=c_med, mode='lines+markers', name='CLOSE Median Return (%)',
                               line=dict(color='#34d399', width=2.5, dash='dash'), marker=dict(size=6)))
    
    fig_c.add_shape(type="line", x0="V+1", x1="V+1", y0=0, y1=1, yref="paper",
                    line=dict(color="#34d399", width=2, dash="dash"))
    fig_c.add_annotation(x="V+1", y=1.02, yref="paper", text="First Sell Day (V+1)",
                         showarrow=False, font=dict(color="#34d399", size=11, family="Inter"))
    fig_c.add_shape(type="line", x0="V+10", x1="V+10", y0=0, y1=1, yref="paper",
                    line=dict(color="#ef4444", width=2.5, dash="dash"))
    fig_c.add_annotation(x="V+10", y=1.02, yref="paper", text="Window Closes (V+10)",
                         showarrow=False, font=dict(color="#ef4444", size=11, family="Inter"))
    fig_c.add_hline(y=0, line_dash="dash", line_color="white")
    
    fig_c.update_layout(
        title=f"MARKET CLOSE Price Trajectory & 95% CI ({tf_title})",
        xaxis_title="Trading Days Post-Vest",
        yaxis_title="Return Relative to V+0 Close (%)",
        template="plotly_dark",
        height=480
    )

    # Add Current Vest Cycle Overlay if available
    cur_cycle = results_data.get('current_vest_cycle')
    if cur_cycle:
        cur_x = [d['day_label'] for d in cur_cycle['days']]
        cur_open_y = [d['open_pct'] for d in cur_cycle['days']]
        cur_close_y = [d['close_pct'] for d in cur_cycle['days']]
        
        fig_o.add_trace(go.Scatter(
            x=cur_x, y=cur_open_y, mode='lines+markers',
            name='🔥 Current Cycle (Aug 2026) OPEN',
            line=dict(color='#ff0055', width=3.5),
            marker=dict(size=9, symbol='diamond', color='#ff0055')
        ))
        
        fig_c.add_trace(go.Scatter(
            x=cur_x, y=cur_close_y, mode='lines+markers',
            name='🔥 Current Cycle (Aug 2026) CLOSE',
            line=dict(color='#ff0055', width=3.5),
            marker=dict(size=9, symbol='diamond', color='#ff0055')
        ))

    fig_o.show()
    fig_c.show()

plot_open_close_separate('all', 'Since 2014 (All-Time)')


## 3. Timeframe Comparison: Since 2014 vs. Last 10 Years vs. Last 5 Years

How has the post-vest dynamics evolved in recent years? We compare performance across the three timeframes.


In [4]:
tf_keys = [('all', 'Since 2014', '#00f2fe'), ('last10y', 'Last 10 Years', '#34d399'), ('last5y', 'Last 5 Years', '#a855f7')]

fig_comp = go.Figure()

for tf_k, tf_lbl, color_hex in tf_keys:
    days_d = results_data[tf_k]['summary_days']
    days_x = [d['day_label'] for d in days_d]
    c_med = [d['close_median'] for d in days_d]
    
    fig_comp.add_trace(go.Scatter(
        x=days_x, y=c_med, mode='lines+markers',
        name=f"{tf_lbl} (n={results_data[tf_k]['n_events']})",
        line=dict(color=color_hex, width=3),
        marker=dict(size=7)
    ))

fig_comp.add_shape(type="line", x0="V+1", x1="V+1", y0=0, y1=1, yref="paper",
                   line=dict(color="#34d399", width=2, dash="dash"))
fig_comp.add_annotation(x="V+1", y=1.02, yref="paper", text="First Sell Day (V+1)",
                        showarrow=False, font=dict(color="#34d399", size=11, family="Inter"))

fig_comp.add_shape(type="line", x0="V+10", x1="V+10", y0=0, y1=1, yref="paper",
                   line=dict(color="#ef4444", width=2.5, dash="dash"))
fig_comp.add_annotation(x="V+10", y=1.02, yref="paper", text="Window Closes (V+10)",
                        showarrow=False, font=dict(color="#ef4444", size=11, family="Inter"))
fig_comp.add_hline(y=0, line_dash="dash", line_color="white")

fig_comp.update_layout(
    title="Timeframe Comparison: CLOSE Median Return Trajectory (% Relative to V+0)",
    xaxis_title="Trading Days Post-Vest",
    yaxis_title="Median Return (%)",
    template="plotly_dark",
    height=520
)
fig_comp.show()


## 4. Historical Regime Shift: Rolling 4-Quarter Ratio of $P_{V+1} / P_{V+10}$ Over Time

To track whether selling early on $V+1$ has become superior to holding until $V+10$ over time, we plot the ratio of:
$$\text{Ratio} = \frac{\text{Price}(V+1\text{ Open})}{\text{Price}(V+10\text{ Close})}$$

- **Ratio $> 1.00$ (Green Shaded Region)**: $V+1$ Open price was **HIGHER** than $V+10$ Close price (selling early on $V+1$ WON).
- **Ratio $< 1.00$ (Red Shaded Region)**: $V+10$ Close price was **HIGHER** than $V+1$ Open price (holding to $V+10$ WON).


In [5]:
# Rolling 4-Quarter Ratio Chart
rData = results_data['all']['ratios_series']
rLabels = [r.get('month_label', r['quarter']) for r in rData]
rRaw = [r['ratio_o1_c10'] for r in rData]
rRoll = [r['rolling_4q_o1_c10'] for r in rData]

yMin = min(rRaw) - 0.02
yMax = max(rRaw) + 0.02

fig_ratio = go.Figure()

fig_ratio.add_trace(go.Scatter(
    x=rLabels, y=rRaw, mode='lines+markers',
    name='Quarterly Ratio P(V+1 Open) / P(V+10 Close)',
    line=dict(color='rgba(96, 165, 250, 0.65)', width=1.5, dash='dot'),
    marker=dict(size=5, color='#60a5fa')
))

fig_ratio.add_trace(go.Scatter(
    x=rLabels, y=rRoll, mode='lines+markers',
    name='Rolling 4-Quarter Avg Ratio',
    line=dict(color='#00f2fe', width=3.5),
    marker=dict(size=7, color='#00f2fe')
))

fig_ratio.add_trace(go.Scatter(
    x=rLabels, y=[1.0]*len(rLabels), mode='lines',
    name='1.0 Parity Line (V+1 = V+10)',
    line=dict(color='#fbbf24', dash='dash', width=2)
))

fig_ratio.update_layout(
    title="Historical Price Ratio P(V+1 Open) / P(V+10 Close) Over Time (2014-2026)",
    xaxis_title="Vest Month & Year",
    yaxis_title="Price Ratio P(V+1 Open) / P(V+10 Close)",
    template="plotly_dark",
    height=520,
    margin=dict(t=50, r=30, l=65, b=85),
    legend=dict(orientation="h", y=1.12, x=0),
    xaxis=dict(tickangle=-45)
)
fig_ratio.show()


## 5. Market Baseline: S&P 500 (SPY) Trajectory vs META (Normalized to V+0 Close)

To test whether broader stock market movements account for post-vest price dips or if the decline is META-specific supply drag, we plot the average price trajectory of the **S&P 500 (SPY)** over the exact same 20-day trading window, normalized to SPY's $V+0$ Close price on the exact same vest dates.


In [6]:
# SPY Baseline Trajectory Chart vs META (with 95% CI)
days_d = results_data['last5y']['summary_days']
days_x = [d['day_label'] for d in days_d]
spy_c_mean = [d['spy_close_mean'] for d in days_d]
spy_c_med = [d['spy_close_median'] for d in days_d]
spy_c_lower = [d['spy_close_ci_lower'] for d in days_d]
spy_c_upper = [d['spy_close_ci_upper'] for d in days_d]
meta_c_mean = [d['close_mean'] for d in days_d]

fig_spy = go.Figure()

fig_spy.add_trace(go.Scatter(
    x=days_x + days_x[::-1],
    y=spy_c_upper + spy_c_lower[::-1],
    fill='toself',
    fillcolor='rgba(168, 85, 247, 0.15)',
    line=dict(color='rgba(0,0,0,0)'),
    name='SPY 95% CI (SE Band)',
    showlegend=True
))

fig_spy.add_trace(go.Scatter(
    x=days_x, y=spy_c_mean, mode='lines+markers',
    name='S&P 500 (SPY) CLOSE Mean Return (%)',
    line=dict(color='#a855f7', width=3),
    marker=dict(size=7)
))

fig_spy.add_trace(go.Scatter(
    x=days_x, y=spy_c_med, mode='lines+markers',
    name='S&P 500 (SPY) CLOSE Median Return (%)',
    line=dict(color='#c084fc', width=2.5, dash='dash'),
    marker=dict(size=6)
))

fig_spy.add_trace(go.Scatter(
    x=days_x, y=meta_c_mean, mode='lines+markers',
    name='META CLOSE Mean Return (%) [Comparison]',
    line=dict(color='#ef4444', width=2.5, dash='dot'),
    marker=dict(size=6)
))

fig_spy.add_shape(type="line", x0="V+1", x1="V+1", y0=0, y1=1, yref="paper",
                  line=dict(color="#34d399", width=2, dash="dash"))
fig_spy.add_annotation(x="V+1", y=1.02, yref="paper", text="First Sell Day (V+1)",
                       showarrow=False, font=dict(color="#34d399", size=11, family="Inter"))

fig_spy.add_shape(type="line", x0="V+10", x1="V+10", y0=0, y1=1, yref="paper",
                  line=dict(color="#ef4444", width=2.5, dash="dash"))
fig_spy.add_annotation(x="V+10", y=1.02, yref="paper", text="Window Closes (V+10)",
                       showarrow=False, font=dict(color="#ef4444", size=11, family="Inter"))
fig_spy.add_hline(y=0, line_dash="dash", line_color="white")

fig_spy.update_layout(
    title="Market Baseline: S&P 500 (SPY) Trajectory vs META (Last 5 Years 2021-2026)",
    xaxis_title="Trading Days Post-Vest",
    yaxis_title="Return Relative to V+0 Close (%)",
    template="plotly_dark",
    height=520,
    margin=dict(t=50, r=30, l=65, b=50),
    legend=dict(orientation="h", y=1.12, x=0)
)
fig_spy.show()


## 6. Final Strategic Takeaways for META Employees (Window Closes at V+10)

Given the constraint that the employee trading window **closes at $V+10$**, here is the optimal execution roadmap:

### 1. Market Open on $V+1$ ($O_1$) is the WORST moment to execute early ❌
- Across all timeframes, selling at Market Open on $V+1$ incurs immediate opening drag:
  - **Since 2014**: **-0.56% mean**, **34.0% win rate**.
  - **Last 10Y**: **-0.64% mean**, **33.3% win rate**.
  - **Last 5Y**: **-0.81% mean**, **31.8% win rate**.

### 2. The Post-Vest Dip ($V+2$ to $V+5$) ❌
- Employee sell orders push prices down to a trough between $V+2$ and $V+3$.
- In the Last 5 Years, Day $V+3$ Open win rate is only **22.7%**!

### 3. The Optimal Execution Window: Day $V+10$ Close ($C_{10}$) ⭐
- As the trading window draws to a close on **Day $V+10$**, supply drag clears and prices rebound strongly:
  - **Since 2014**: Day $V+10$ Close median is **+0.96%** (Win Rate **56.0%**).
  - **Last 10 Years**: Day $V+10$ Close median is **+1.08%** (Win Rate **57.1%**).
  - **Last 5 Years**: Day $V+10$ Close median is **+1.45%** (Win Rate **59.1%**).

---

### Optimal Execution Decision Matrix (Trading Window $V+0$ to $V+10$):

| Trading Day | Execution Moment | Since 2014 Median | Last 10Y Median | Last 5Y Median | Win Rate ($V+0$) | Strategic Recommendation |
| :---: | :--- | :---: | :---: | :---: | :---: | :--- |
| **$V+0$** | **Vest Day Close** | 0.00% | 0.00% | 0.00% | Baseline | Reference Price |
| **$V+1$** | **Market Open ($O_1$)** | -0.42% | -0.47% | -0.68% | 34.0% | ❌ **Worst Early Execution** (Opening order drag) |
| **$V+1$** | **Market Close ($C_1$)** | -0.38% | -0.40% | -0.55% | 40.0% | 🛡️ Slightly better than Open if forced to sell Day 1 |
| **$V+2$** | **Market Open ($O_2$)** | -0.82% | -0.89% | -1.12% | 26.0% | ❌ Peak early supply drag |
| **$V+3$** | **Market Close ($C_3$)** | -1.50% | -1.58% | -1.72% | 38.0% | ❌ Peak price trough |
| **$V+5$** | **Market Close ($C_5$)** | -0.31% | -0.35% | -0.42% | 46.0% | ⚠️ Rebound beginning |
| **$V+10$** | **Market Close ($C_{10}$)** | **+0.96%** | **+1.08%** | **+1.45%** | **56.0%–59.1%** | ⭐ **OPTIMAL WINDOW CLOSE EXECUTION** |
